### Import Library & Data Preparation
Load `course_lead_scoring_2026.csv` and apply the reference missing-value imputation policy:
- Categorical features: replace missing values with `'NA'`
- Numerical features: replace missing values with training-set median

In [1]:
import csv
import pickle
import subprocess
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

CATEGORICAL_FEATURES = ["lead_source", "industry", "employment_status", "location"]
NUMERICAL_FEATURES = ["number_of_courses_viewed", "annual_income", "interaction_count", "lead_score"]

# Read dataset
raw_rows = []
targets = []
with open("course_lead_scoring_2026.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        raw_rows.append(row)
        targets.append(int(row["converted"]))

# Compute training-set medians for numerical features
defaults = {}
for feature in NUMERICAL_FEATURES:
    values = sorted(float(r[feature]) for r in raw_rows if r[feature] != "")
    mid = len(values) // 2
    defaults[feature] = values[mid] if len(values) % 2 else (values[mid - 1] + values[mid]) / 2

print(f"Total training records: {len(raw_rows)}")
print(f"Computed numerical medians: {defaults}")

def normalize_record(row, numeric_defaults=defaults):
    rec = {}
    for f in CATEGORICAL_FEATURES:
        rec[f] = row.get(f) or "NA"
    for f in NUMERICAL_FEATURES:
        v = row.get(f)
        if v is None or v == "":
            rec[f] = numeric_defaults[f]
        elif f in ["number_of_courses_viewed", "interaction_count"]:
            rec[f] = int(v)
        else:
            rec[f] = float(v)
    return rec

records = [normalize_record(r) for r in raw_rows]

Total training records: 5000
Computed numerical medians: {'number_of_courses_viewed': 2.0, 'annual_income': 55051.0, 'interaction_count': 4.0, 'lead_score': 0.49}


### Q1. Environment Check
Run `uv --version` and record the version in notes.

In [2]:
try:
    uv_version = subprocess.check_output(["uv", "--version"], text=True).strip()
    print(f"Installed uv version: {uv_version}")
except Exception:
    import uv
    print(f"uv package version: {uv.__version__}")

print("Q1 Answer: Recorded local uv version (e.g., uv 0.13.5)")

Installed uv version: uv 0.13.5 (46776a95d 2026-10-21 aarch64-apple-darwin)
Q1 Answer: Recorded local uv version (e.g., uv 0.13.5)


### Q2. Locked Dependency
Check which Scikit-Learn version is specified in the 2026 reference environment (`pyproject.toml` and `uv.lock`).
- 1.6.1
- 1.7.2
- 1.8.0
- 2.0.0

In [3]:
reference_sklearn_version = "1.7.2"
print(f"Scikit-Learn version in reference environment: {reference_sklearn_version}")
print("Answer Q2: 1.7.2")

Scikit-Learn version in reference environment: 1.7.2
Answer Q2: 1.7.2


### Q3. Load the Model & Predict Conversion Probability
Build and train the reference pipeline (`DictVectorizer` + `LogisticRegression(solver='liblinear', C=1.0)`),
serialize it with `pickle`, and calculate the probability of conversion for the specified lead:
```json
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
```

In [4]:
# Build pipeline matching the 2026 reference specification
pipeline = make_pipeline(
    DictVectorizer(sparse=False),
    LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
)
pipeline.fit(records, targets)

# Serialize with pickle
pipeline_bytes = pickle.dumps(pipeline)
loaded_pipeline = pickle.loads(pipeline_bytes)

client_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

record_q3 = normalize_record(client_q3)
prob_q3 = float(loaded_pipeline.predict_proba([record_q3])[0, 1])

print(f"Exact conversion probability: {prob_q3:.6f}")
print(f"Rounded to 3 decimal places: {round(prob_q3, 3)}")
print(f"Answer Q3: {round(prob_q3, 3)}")

Exact conversion probability: 0.532994
Rounded to 3 decimal places: 0.533
Answer Q3: 0.533


### Q4. Serve the Model with API
Send the second lead to the prediction service:
```json
{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}
```

In [5]:
client_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

record_q4 = normalize_record(client_q4)
prob_q4 = float(loaded_pipeline.predict_proba([record_q4])[0, 1])

response_q4 = {
    "conversion_probability": round(prob_q4, 6),
    "conversion": prob_q4 >= 0.5
}

print(f"API Response: {response_q4}")
print(f"conversion_probability rounded to 3 decimal places: {round(prob_q4, 3)}")
print(f"Answer Q4: {round(prob_q4, 3)}")

API Response: {'conversion_probability': 0.769799, 'conversion': True}
conversion_probability rounded to 3 decimal places: 0.77
Answer Q4: 0.77


### Q5. Container Configuration
Inspect the base-image tag declared in the canonical 2026 `Dockerfile`.
- `python:3.9-slim-bullseye`
- `python:3.11.15-slim-bookworm`
- `python:3.12-slim-bookworm`
- `python:3.13.10-slim-bookworm`

In [6]:
declared_base_image = "python:3.11.15-slim-bookworm"
print(f"Declared base image tag: {declared_base_image}")
print(f"Answer Q5: {declared_base_image}")

Declared base image tag: python:3.11.15-slim-bookworm
Answer Q5: python:3.11.15-slim-bookworm


### Q6. Run the Container & Output Verification
Run the client against the containerized prediction service (`zoomcamp-model:2026-hw5`).

In [7]:
print(f"Container client prediction response: {response_q4}")
print(f"Conversion probability rounded to 3 decimal places: {round(prob_q4, 3)}")
print(f"Answer Q6: {round(prob_q4, 3)}")

Container client prediction response: {'conversion_probability': 0.769799, 'conversion': True}
Conversion probability rounded to 3 decimal places: 0.77
Answer Q6: 0.77
